# Aletheia's Quest submission
Entry point: a stable `harness` runs the method module selected by `ALETHEIA_METHOD` (default set in the next cell; on the leaderboard no env is set, so the default ships). The last cell guarantees a valid `submission.csv` is always written.

## Method report — `dyad_plainfuse` (tag black)

This notebook runs exactly one detector module — **`dyad_plainfuse`**, selected by `ALETHEIA_METHOD`
below. The same detection logic runs for every model organism and every dataset: there is no
per-organism branching anywhere on the call path.

The report is kept beside this notebook rather than inside it, so that it cannot drift from the
method that was actually fired:

| document | contents |
|---|---|
| `submission/README.md` | what `dyad_plainfuse` does, channel by channel, and why the tag is **black** |
| `submission/FAIRNESS.md` | rule-by-rule compliance, cited to `file:line` inside this bundle |
| `submission/VERSIONS.md` | what changed, and when |

Earlier revisions of this notebook repeated the method description here. That duplicate is
deliberately gone — it was the thing that went stale.


## Training reproducibility

Whether `dyad_plainfuse` loads any trained artifact, which one, and the code that regenerates it are
stated in `submission/README.md` and, where artifacts are involved, `submission/training/PROVENANCE.md`.

This repository also ships artifacts belonging to *other* methods it contains; those are inert
unless the armed method loads them. The next cell reproduces the bundled artifacts on demand and is
a no-op during scoring.


In [ ]:
# --- Training reproducibility (competition rule: training contained in the submission and
# reproducible when organizers re-run it). This cell is a NO-OP during scoring. ---
# Organizers: set ALETHEIA_RETRAIN=1 to reproduce the bundled trained artifacts (deterministic,
# numpy-only, asserts an exact match to what ships), or ALETHEIA_RETRAIN=scratch to also re-derive
# the NDIF-dependent leg (needs .env + network). Which of these -- if any -- the armed method
# `dyad_plainfuse` actually loads is stated in submission/README.md; the chain is in
# submission/training/PROVENANCE.md.
import os as _os, subprocess as _sp, sys as _sys
_retrain = _os.environ.get("ALETHEIA_RETRAIN", "")
if _retrain and _os.path.isdir("submission/training"):
    _cmd = [_sys.executable, "submission/training/rebuild_pm22.py"]
    if _retrain == "scratch":
        _cmd.append("--from-scratch")            # NDIF re-derivation may not be bit-identical
    else:
        _cmd += ["--verify", "submission/pm22.npz"]  # deterministic: assert exact reproduction
    print(f"[submission] ALETHEIA_RETRAIN={_retrain} -> rebuilding pm22.npz from submission/training/")
    _sp.run(_cmd, check=True)
else:
    print("[submission] scoring run: bundled artifacts used as shipped; which ones (if any) "
          "`dyad_plainfuse` loads is stated in submission/README.md. "
          "Set ALETHEIA_RETRAIN=1 to reproduce them (see submission/training/PROVENANCE.md).")

In [ ]:
import os, sys
sys.path.insert(0, "submission")
import util

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
METHOD = os.environ.get("ALETHEIA_METHOD", "dyad_plainfuse")   # default shipped on the leaderboard

In [ ]:
# On ANY failure still write a valid submission.csv (neutral 0.5) so the run never hard-fails.
import numpy as np, traceback
try:
    import harness, methods
    harness.run_submission(methods.load(METHOD), DATASET_NAME, limit=LIMIT, batch_size=8, max_len=256, remote=True)
except Exception as e:
    traceback.print_exc()
    print(f"[submission] harness failed ({type(e).__name__}: {e}) -> writing 0.5 fallback", flush=True)
    ex = util.load_examples(DATASET_NAME)
    n = len(ex) if LIMIT is None else min(LIMIT, len(ex))
    util.write_submission(list(ex["index"])[:n], np.full(n, 0.5))

In [ ]:
# Ensure submission.csv exists no matter what.
import os as _os, numpy as _np
if not _os.path.exists("submission.csv"):
    _ex = util.load_examples(DATASET_NAME)
    _n = len(_ex) if LIMIT is None else min(LIMIT, len(_ex))
    util.write_submission(list(_ex["index"])[:_n], _np.full(_n, 0.5))
    print("[submission] wrote final 0.5 safety net")